In [1]:
from ultralytics import YOLO
import torch
import os
import numpy as np
import shutil

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.0.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 2050
GPU Memory: 4.29 GB


In [2]:
data_yaml = '../dataset/processed_yolo/data.yaml'

# Buat folder models jika belum ada
os.makedirs('../models', exist_ok=True)
model_target_path = '../models/yolov8n.pt'

# Cek apakah file sudah ada di target
if not os.path.exists(model_target_path):
    print("Mendownload YOLOv8n ke folder models...")
    # Download sementara di current dir
    temp_model = YOLO('yolov8n.pt')  # Akan terdownload di sini
    # Pindahkan ke target
    if os.path.exists('yolov8n.pt'):
        shutil.move('yolov8n.pt', model_target_path)
        print(f"Model disimpan di: {model_target_path}")
    else:
        # Fallback jika download gagal, cari di tempat lain
        raise FileNotFoundError("Gagal mendownload model YOLO")
else:
    print(f"Model sudah ada di: {model_target_path}")

Model sudah ada di: ../models/yolov8n.pt


In [4]:
# Load model dari folder models
model = YOLO(model_target_path)

# Train
results = model.train(
    data=data_yaml,
    epochs=80,
    imgsz=640,
    batch=16,
    nbs=64,                    # Gradient accumulation (cicil) - default 64
    device=0,                  # GPU
    workers=0,                 # Turun jadi 2 agar lebih ringan dan stabil
    amp=False,                 # matikan Aamp untuk menghindari error Numpy
    
    # Augmentasi
    augment=True,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    
    # Early stopping
    patience=20,
    cos_lr=True,
    warmup_epochs=3,
    lr0=0.01,
    
    # Output (akan tetap tersimpan di runs/detect/... )
    project='helmet_detection_runs',
    name='yolov8n_helmet',
    exist_ok=True,
    verbose=True,
)

print("Training Selesai!")

Ultralytics 8.4.70  Python-3.10.8 torch-2.0.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=../dataset/processed_yolo/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=../models/yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_helmet, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

In [5]:
# Evaluasi di validation set
metrics = model.val(data=data_yaml, split='val')
print(f"Validation mAP50: {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

# Evaluasi di test set
metrics_test = model.val(data=data_yaml, split='test')
print(f"Test mAP50: {metrics_test.box.map50:.4f}")
print(f"Test mAP50-95: {metrics_test.box.map:.4f}")

Ultralytics 8.4.70  Python-3.10.8 torch-2.0.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1163.6359.1 MB/s, size: 496.0 KB)
val: Scanning C:\Users\Pongo\Desktop\Belajar\ai-models\helmet-detection\src\dataset\processed_yolo\val\labels.cache... 153 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 153/153  0.0s
val: C:\Users\Pongo\Desktop\Belajar\ai-models\helmet-detection\src\dataset\processed_yolo\val\images\BikesHelmets75.png: 2 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.1it/s 4.8s.4ss
                   all        153        299        0.8      0.815      0.861      0.527
           With helmet        114        217      0.873      0.886      0.917       0.58
        Without helmet         48         82      0.727      0.744      0.804      0.474
Speed: 3.

In [8]:
# Salin best.pt ke folder MODELSAVE
# source_best = 'helmet_detection_runs/yolov8n_helmet/weights/best.pt'
source_best = './runs/detect/helmet_detection_runs/yolov8n_helmet/weights/best.pt'
target_dir = '../models/'
os.makedirs(target_dir, exist_ok=True)

shutil.copy2(source_best, os.path.join(target_dir, 'best.pt'))
print(f"best.pt disimpan di {target_dir}")

# Export ke ONNX
model_onnx = YOLO(source_best)
onnx_path = model_onnx.export(format='onnx', imgsz=640)
shutil.copy2(onnx_path, os.path.join(target_dir, 'best.onnx'))
print(f"best.onnx disimpan di {target_dir}")

# Copy data.yaml
shutil.copy2('../dataset/processed_yolo/data.yaml', os.path.join(target_dir, 'data.yaml'))
print("data.yaml ikut disalin")

best.pt disimpan di ../models/
Ultralytics 8.4.70  Python-3.10.8 torch-2.0.1+cu118 CPU (12th Gen Intel Core i7-12650H)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs\detect\helmet_detection_runs\yolov8n_helmet\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.0 MB)

ONNX: starting export with onnx 1.22.0 opset 17...
============= Diagnostic Run torch.onnx.export version 2.0.1+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  1.8s, saved as 'runs\detect\helmet_detection_runs\yolov8n_helmet\weights\best.onnx' (11.7 MB)

Export complete (2.4s)
Results saved to C:\Users\Pongo\Desktop\Belajar\ai-models\helmet-detection\src\notebooks\runs\detect\helmet_detection_runs\yolov8n_helmet\weights\best.onnx
Predict:         yolo predict task=detec